# Spatial CD204+ M2 macrophage neighborhood graph as a checkpoint-immunotherapy non-response predictor in TFE3-fusion osteosarcoma

_Notebook scaffold generated by deltasci. Canonical workflow code is real; `# TODO` markers indicate where substantive customization is required._
_See `README.md` for orientation._


## Hypothesis

A heterogeneous graph neural network operating on per-tumor single-cell spatial transcriptomics, with cell-type-pair typed edges (in particular, CD204+ M2 macrophage proximity to malignant cells), will predict checkpoint-immunotherapy non-response in osteosarcoma at AUROC ≥ 0.75 on held-out data, improving over a bulk IFN-γ Hallmark signature baseline by ≥ 0.07 AUROC, with calibration intercept |α| < 0.05 and slope within [0.9, 1.1]. Performance is reported stratified by TFE3 fusion status.

### Falsifiability
- **Prediction:** The HGT-with-CD204+M2-typed-edges model achieves a higher AUROC for checkpoint-immunotherapy non-response than the bulk IFN-γ signature baseline on the external held-out OS cohort.
- **Threshold:** AUROC ≥ 0.75 absolute, AND AUROC improvement ≥ 0.07 over bulk IFN-γ baseline with 95% CI lower bound > IFN-γ baseline, AND calibration intercept |α| < 0.05, AND calibration slope in [0.9, 1.1].
- **Null outcome:** AUROC improvement < 0.03 over bulk IFN-γ baseline, OR calibration slope outside [0.85, 1.15], falsifies the hypothesis: the spatial-graph cell-type-pair edge typing did not add value over scalar M2 density or bulk IFN-γ score.


In [ ]:
# === Imports — life-sciences + ML stack ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import anndata as ad
import scanpy as sc
# import squidpy as sq  # uncomment for spatial transcriptomics

from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    confusion_matrix, classification_report,
)
from sklearn.model_selection import StratifiedKFold

import torch
import torch.nn as nn
# import torch_geometric as pyg  # uncomment for graph-based architectures

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, frameon=False)

RANDOM_SEED = 0
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)


## Data acquisition

- **Primary dataset:** Institutional pretreatment FFPE OS biopsy cohort with linked PD-1 / PD-L1 inhibitor RECIST outcomes; Xenium 10x in-situ panel.
- **Accession / URL:** `(institutional cohort — DUA + IRB required; reference OS scRNA-seq atlas at GEO GSE152048 for cell-type marker priors)`
- **Access constraints:** IRB approval, FFPE block availability, RECIST adjudication, multi-site DUA for external validation


In [ ]:
# === Data acquisition ===
PRIMARY_ACCESSION = '(institutional cohort — DUA + IRB required; reference OS scRNA-seq atlas at GEO GSE152048 for cell-type marker priors)'
DATA_PATH = None  # TODO: set local .h5ad path or download URL
CLINICAL_CSV = None  # TODO: set path to clinical metadata CSV (patient_id, outcome)

if DATA_PATH is None:
    raise NotImplementedError(
        f'Set DATA_PATH for accession {PRIMARY_ACCESSION}. '
        'See 06_protocol/protocol.md and 08_audits/citations.json for verification status.'
    )

adata = sc.read_h5ad(DATA_PATH)
print(f'loaded: {adata.n_obs} cells × {adata.n_vars} genes')
print(f'available .obs columns: {list(adata.obs.columns)}')

if CLINICAL_CSV is not None:
    clinical = pd.read_csv(CLINICAL_CSV)
    print(f'clinical: {len(clinical)} patients, {clinical.columns.tolist()}')


## Step 1: Cohort assembly

Pretreatment FFPE biopsies with linked best-RECIST-response and TFE3-fusion status.

**Inputs:** FFPE blocks, clinical chart
**Outputs:** cohort manifest

**Methods cited:**
- RECIST 1.1, Eisenhauer 2009


In [ ]:
# === Step 1: Cohort assembly ===
# Canonical pattern: assemble per-patient cohort with linked outcomes.

OUTCOME_COL = 'response'  # TODO: rename to your outcome column
PATIENT_ID_COL = 'patient_id'  # TODO: rename if your column differs

if CLINICAL_CSV is None:
    raise NotImplementedError('Set CLINICAL_CSV in the data-acquisition cell.')

clinical = pd.read_csv(CLINICAL_CSV)
assert OUTCOME_COL in clinical.columns, f'Outcome column {OUTCOME_COL!r} missing'
assert PATIENT_ID_COL in clinical.columns, f'Patient-ID column {PATIENT_ID_COL!r} missing'

n = len(clinical)
print(f'cohort size: {n} patients')
if clinical[OUTCOME_COL].dtype != object:
    print(f'positive-class rate: {(clinical[OUTCOME_COL] == 1).mean():.1%}')

if PATIENT_ID_COL in adata.obs.columns:
    adata.obs = adata.obs.merge(
        clinical[[PATIENT_ID_COL, OUTCOME_COL]],
        on=PATIENT_ID_COL, how='left',
    )
    print(f'cells with linked outcome: {adata.obs[OUTCOME_COL].notna().sum()}/{adata.n_obs}')
else:
    print(f'WARNING: adata.obs has no {PATIENT_ID_COL!r}; merge manually before continuing.')


## Step 2: Spatial transcriptomics + QC

Run Xenium panel; QC per cell, per FOV, per sample.

**Inputs:** FFPE sections
**Outputs:** per-cell expression matrices

**Methods cited:**
- github.com/scverse/scanpy
- github.com/scverse/squidpy


In [ ]:
# === Step 2: Spatial transcriptomics + QC ===
# Canonical scanpy QC: per-cell + per-gene filters, normalization, log1p.

MIN_GENES_PER_CELL = 200  # TODO: tune for your platform (Xenium ~30, scRNA ~200-500)
MIN_CELLS_PER_GENE = 3
MAX_PCT_MITO = 20.0       # TODO: tune (15-25% is common for fresh tissue)
TARGET_SUM = 1e4

n_pre = adata.n_obs

mt_prefix = 'MT-' if any(g.startswith('MT-') for g in adata.var_names) else 'mt-'
adata.var['mt'] = adata.var_names.str.startswith(mt_prefix)
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True, percent_top=None, log1p=False)

sc.pp.filter_cells(adata, min_genes=MIN_GENES_PER_CELL)
sc.pp.filter_genes(adata, min_cells=MIN_CELLS_PER_GENE)
adata = adata[adata.obs['pct_counts_mt'] < MAX_PCT_MITO, :].copy()

sc.pp.normalize_total(adata, target_sum=TARGET_SUM)
sc.pp.log1p(adata)

n_post = adata.n_obs
print(f'QC: {n_pre} → {n_post} cells ({100*(n_pre-n_post)/n_pre:.1f}% removed)')
print(f'genes retained: {adata.n_vars}')

adata.layers['log1p'] = adata.X.copy()


## Step 3: Cell-type annotation

Reference-mapped annotation against OS scRNA-seq atlas; tag CD204+ M2, malignant, T-cell, fibroblast.

**Inputs:** per-cell expression
**Outputs:** typed cells

**Methods cited:**
- GEO GSE152048 marker panel


In [ ]:
# === Step 3: Cell-type annotation ===
# Canonical scanpy: HVG → PCA → neighbors → UMAP → leiden, then marker scoring.

N_TOP_GENES = 2000
N_PCS = 50
N_NEIGHBORS = 15
LEIDEN_RESOLUTION = 0.5  # TODO: tune (0.3-0.8 typical)

sc.pp.highly_variable_genes(adata, n_top_genes=N_TOP_GENES, flavor='seurat')
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=N_PCS)
sc.pp.neighbors(adata, n_neighbors=N_NEIGHBORS, n_pcs=min(N_PCS, 30))
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=LEIDEN_RESOLUTION)

print(f'leiden clusters: {adata.obs["leiden"].nunique()}')

# Marker-score-based cell-type calling. EDIT this dictionary for your panels.
MARKER_PANELS = {
    'M2_macrophage': ['CD204', 'CD163', 'MSR1', 'MARCO', 'MRC1'],
    'M1_macrophage': ['CD68', 'NOS2', 'CXCL10', 'CXCL11'],
    'T_cell':        ['CD3D', 'CD3E', 'CD8A', 'CD4'],
    'NK_cell':       ['NCAM1', 'NKG7', 'GNLY'],
    'fibroblast':    ['COL1A1', 'DCN', 'PDGFRA'],
    'endothelial':   ['PECAM1', 'VWF', 'CDH5'],
    'malignant':     ['SATB2', 'RUNX2'],  # TODO: substitute markers for your tumor type
}

score_cols = []
for cell_type, markers in MARKER_PANELS.items():
    available = [g for g in markers if g in adata.var_names]
    if available:
        sc.tl.score_genes(adata, available, score_name=f'{cell_type}_score')
        score_cols.append(f'{cell_type}_score')
        print(f'{cell_type}: scored using {len(available)}/{len(markers)} markers')
    else:
        print(f'{cell_type}: WARNING — no markers in panel; skipping')

if score_cols:
    scores_df = adata.obs[score_cols]
    adata.obs['celltype'] = scores_df.idxmax(axis=1).str.replace('_score', '', regex=False)
    adata.obs['celltype_confidence'] = scores_df.max(axis=1) - scores_df.median(axis=1)
    print(f'cell-type calls: {adata.obs["celltype"].value_counts().to_dict()}')


## Step 4: Spatial graph construction

k-NN in physical-µm with edge type = cell-type pair (M2-malignant, M2-T, T-malignant, ...).

**Inputs:** typed cells, x/y coords
**Outputs:** per-tumor heterogeneous graph

**Methods cited:**
- github.com/pyg-team/pytorch_geometric


In [ ]:
# === Step 4: Spatial graph construction ===
# Canonical scanpy: HVG → PCA → neighbors → UMAP → leiden, then marker scoring.

N_TOP_GENES = 2000
N_PCS = 50
N_NEIGHBORS = 15
LEIDEN_RESOLUTION = 0.5  # TODO: tune (0.3-0.8 typical)

sc.pp.highly_variable_genes(adata, n_top_genes=N_TOP_GENES, flavor='seurat')
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=N_PCS)
sc.pp.neighbors(adata, n_neighbors=N_NEIGHBORS, n_pcs=min(N_PCS, 30))
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=LEIDEN_RESOLUTION)

print(f'leiden clusters: {adata.obs["leiden"].nunique()}')

# Marker-score-based cell-type calling. EDIT this dictionary for your panels.
MARKER_PANELS = {
    'M2_macrophage': ['CD204', 'CD163', 'MSR1', 'MARCO', 'MRC1'],
    'M1_macrophage': ['CD68', 'NOS2', 'CXCL10', 'CXCL11'],
    'T_cell':        ['CD3D', 'CD3E', 'CD8A', 'CD4'],
    'NK_cell':       ['NCAM1', 'NKG7', 'GNLY'],
    'fibroblast':    ['COL1A1', 'DCN', 'PDGFRA'],
    'endothelial':   ['PECAM1', 'VWF', 'CDH5'],
    'malignant':     ['SATB2', 'RUNX2'],  # TODO: substitute markers for your tumor type
}

score_cols = []
for cell_type, markers in MARKER_PANELS.items():
    available = [g for g in markers if g in adata.var_names]
    if available:
        sc.tl.score_genes(adata, available, score_name=f'{cell_type}_score')
        score_cols.append(f'{cell_type}_score')
        print(f'{cell_type}: scored using {len(available)}/{len(markers)} markers')
    else:
        print(f'{cell_type}: WARNING — no markers in panel; skipping')

if score_cols:
    scores_df = adata.obs[score_cols]
    adata.obs['celltype'] = scores_df.idxmax(axis=1).str.replace('_score', '', regex=False)
    adata.obs['celltype_confidence'] = scores_df.max(axis=1) - scores_df.median(axis=1)
    print(f'cell-type calls: {adata.obs["celltype"].value_counts().to_dict()}')


## Step 5: HGT encoder + survival head

Heterogeneous Graph Transformer (Hu et al 2020) → graph readout → MLP head; multitask loss.

**Inputs:** graphs, outcomes
**Outputs:** trained model, predictions

**Methods cited:**
- Hu et al 2020 WWW HGT


In [ ]:
# === Step 5: HGT encoder + survival head ===
# GNN + classification head — skeleton with a real training loop.
# The forward pass is real (GAT-based); swap to HGT for typed edges if your
# hypothesis depends on edge typing.

import torch_geometric.nn as pyg_nn
from torch_geometric.loader import DataLoader

HIDDEN_DIM = 64
NUM_HEADS = 4
NUM_LAYERS = 2
DROPOUT = 0.2
BATCH_SIZE = 4   # graphs per batch; small for whole-tumor graphs
EPOCHS = 30
LR = 1e-3

class TumorClassifier(nn.Module):
    def __init__(self, in_dim, hidden=HIDDEN_DIM, num_layers=NUM_LAYERS):
        super().__init__()
        self.convs = nn.ModuleList([
            pyg_nn.GATConv(in_dim if i == 0 else hidden, hidden,
                           heads=NUM_HEADS, concat=False, dropout=DROPOUT)
            for i in range(num_layers)
        ])
        self.pool = pyg_nn.global_mean_pool
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(hidden, 1),
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for conv in self.convs:
            x = torch.relu(conv(x, edge_index))
        h = self.pool(x, batch)
        return self.head(h).squeeze(-1)

# TODO: assemble dataset (tumor_graphs from previous step) with patient labels
labeled_graphs = []  # list[Data] with .y attached
if not labeled_graphs:
    raise NotImplementedError(
        'Attach binary outcome labels to each per-tumor graph: graph.y = torch.tensor([label])'
    )

loader = DataLoader(labeled_graphs, batch_size=BATCH_SIZE, shuffle=True)
in_dim = labeled_graphs[0].x.shape[1]
model = TumorClassifier(in_dim=in_dim)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.BCEWithLogitsLoss()

model.train()
for epoch in range(EPOCHS):
    losses = []
    for batch in loader:
        optimizer.zero_grad()
        logits = model(batch)
        loss = loss_fn(logits, batch.y.float())
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    if epoch % 5 == 0:
        print(f'epoch {epoch}: train loss = {np.mean(losses):.4f}')


## Step 6: Evaluation + calibration

AUROC + AUPRC + decision-curve analysis vs IFN-γ Hallmark baseline; stratified by TFE3 status.

**Inputs:** predictions
**Outputs:** metrics, figures

**Methods cited:**
- TRIPOD 2015


In [ ]:
# === Step 6: Evaluation + calibration ===
# Held-out evaluation: AUROC + AUPRC + Brier + calibration curve.

model.eval()
y_true_list, y_pred_list = [], []
with torch.no_grad():
    for batch in loader:  # TODO: replace with held-out validation loader
        logits = model(batch)
        probs = torch.sigmoid(logits).cpu().numpy()
        y_pred_list.extend(probs.tolist())
        y_true_list.extend(batch.y.cpu().numpy().tolist())
y_true = np.array(y_true_list)
y_pred = np.array(y_pred_list)

model_auroc = roc_auc_score(y_true, y_pred)
model_auprc = average_precision_score(y_true, y_pred)
model_brier = brier_score_loss(y_true, y_pred)
print(f'AUROC: {model_auroc:.4f}')
print(f'AUPRC: {model_auprc:.4f}')
print(f'Brier: {model_brier:.4f}')

from sklearn.calibration import calibration_curve
prob_true, prob_pred = calibration_curve(y_true, y_pred, n_bins=10, strategy='quantile')
for pt, pp in zip(prob_true, prob_pred):
    print(f'  bin midpoint {pp:.2f} → observed rate {pt:.2f}')

# TODO: compute baseline predictions (e.g., bulk IFN-γ score per patient).
y_pred_baseline = None
if y_pred_baseline is None:
    raise NotImplementedError(
        'Compute baseline predictions for the same held-out cohort.'
    )
baseline_auroc = roc_auc_score(y_true, y_pred_baseline)
print(f'baseline AUROC: {baseline_auroc:.4f}')
print(f'lift over baseline: {model_auroc - baseline_auroc:+.4f}')


## Falsifiability check


In [ ]:
# === Falsifiability check ===
# Primary metric: AUROC for predicting checkpoint-inhibitor non-response (best RECIST = PD)
# Success threshold: AUROC ≥ 0.75 absolute AND ≥ 0.07 above bulk IFN-γ baseline AUROC, with calibration intercept |α| < 0.05 and slope ∈ [0.9, 1.1]
# Null outcome:     AUROC improvement < 0.02 over bulk IFN-γ baseline, OR calibration slope outside [0.85, 1.15], OR no detectable signal in TFE3-fusion stratum

try:
    model_metric_value = float(model_auroc)
    baseline_metric_value = float(baseline_auroc)
except NameError:
    model_metric_value = None
    baseline_metric_value = None

if model_metric_value is None or baseline_metric_value is None:
    raise NotImplementedError('Run the evaluation step first, or set values manually.')

lift = model_metric_value - baseline_metric_value
print(f'primary metric (model):    {model_metric_value:.4f}')
print(f'primary metric (baseline): {baseline_metric_value:.4f}')
print(f'lift over baseline:        {lift:+.4f}')

MIN_LIFT_FOR_HYPOTHESIS = 0.07  # TODO: align with the falsifiability threshold above
assert lift >= MIN_LIFT_FOR_HYPOTHESIS, (
    f'Lift {lift:+.4f} is below the falsifiability threshold {MIN_LIFT_FOR_HYPOTHESIS}. '
    'Per the hypothesis null outcome, this run falsifies the hypothesis.'
)
print('falsifiability check PASSED')


## Notes

When you've filled in the TODOs and the falsifiability check passes:

1. Re-audit with `deltasci audit <run-dir> --write` so the citation audit
   reflects any new datasets or references you wired in.
2. Consider archiving and starting a fresh deltasci pass with `deltasci run
   --iterate-on <this-run-dir>` so the iteration history captures the change.
